In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
%matplotlib inline

from src.utils import set_seed
set_seed(42)
print('Setup complete')

## Data Loading

In [ ]:
from src.data_loader import load_data

X_raw, y, ch_names = load_data(
    data_path='../data/raw/synthetic_eeg.csv',
    use_synthetic=True,
    n_subjects=60
)

print(f'X_raw shape : {X_raw.shape}  → (subjects, channels, samples)')
print(f'Labels      : {y.shape}  → Healthy={( y==0).sum()}, Depressed={(y==1).sum()}')
print(f'Channels    : {ch_names[:5]} … (total {len(ch_names)})')

## Raw EEG Signal Visualization

In [ ]:
fs = 256  # Hz
t  = np.arange(X_raw.shape[-1]) / fs

fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
fig.suptitle('Raw EEG Signal — Subject 0 vs Subject 30 (ch_0)', fontsize=13, fontweight='bold')

axes[0].plot(t, X_raw[0, 0], color='#3498db', lw=0.6, alpha=0.9)
axes[0].set_title('Healthy Subject (idx 0)', fontsize=11)
axes[0].set_ylabel('Amplitude (µV)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, X_raw[30, 0], color='#e74c3c', lw=0.6, alpha=0.9)
axes[1].set_title('Depressed Subject (idx 30)', fontsize=11)
axes[1].set_ylabel('Amplitude (µV)')
axes[1].set_xlabel('Time (s)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Preprocessing

In [ ]:
from src.preprocessing import preprocess_signals

X_proc = preprocess_signals(X_raw, fs=256.0)

# Plot comparison
fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
fig.suptitle('EEG Preprocessing — Subject 0, Channel 0', fontsize=13, fontweight='bold')

axes[0].plot(t, X_raw[0, 0],  color='#e74c3c', lw=0.6, label='Raw')
axes[0].set_title('Raw Signal'); axes[0].set_ylabel('µV'); axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, X_proc[0, 0], color='#2ecc71', lw=0.6, label='Filtered + Normalized')
axes[1].set_title('After Bandpass + Z-score'); axes[1].set_ylabel('z-score')
axes[1].set_xlabel('Time (s)'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Feature Extraction

In [ ]:
from src.feature_extraction import extract_features

X_features, y_feat, feat_names = extract_features(X_proc, y, fs=256.0, ch_names=ch_names)

print(f'Feature matrix: {X_features.shape}')
print(f'Sample features: {feat_names[:6]}')

df_feat = pd.DataFrame(X_features, columns=feat_names)
df_feat['label'] = y_feat

# Compare mean feature values between classes
comparison = df_feat.groupby('label')[feat_names[:10]].mean().T
comparison.columns = ['Healthy', 'Depressed']
comparison['Difference'] = comparison['Depressed'] - comparison['Healthy']
comparison.head(10)

In [ ]:
# Band power comparison - clinically meaningful features
band_feats = ['ch_0_delta_power', 'ch_0_theta_power', 'ch_0_alpha_power', 'ch_0_beta_power']
available  = [f for f in band_feats if f in feat_names]

fig, axes = plt.subplots(1, len(available), figsize=(14, 4))
if len(available) == 1:
    axes = [axes]
    
colors = {'Healthy': '#3498db', 'Depressed': '#e74c3c'}

for ax, feat in zip(axes, available):
    feat_idx = feat_names.index(feat)
    for label_val, color, label_name in [(0, '#3498db', 'Healthy'), (1, '#e74c3c', 'Depressed')]:
        mask = y_feat == label_val
        ax.hist(X_features[mask, feat_idx], bins=15, alpha=0.65, color=color, label=label_name, density=True)
    ax.set_title(feat.replace('ch_0_','').replace('_',' ').title(), fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('EEG Band Power Distributions (Channel 0)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Spike Encoding

In [ ]:
from src.spike_encoding import rate_encode

T = 50
spikes = rate_encode(X_features, T=T, seed=42)
print(f'Spikes shape: {spikes.shape}  → (T={T}, samples={spikes.shape[1]}, features={spikes.shape[2]})')
print(f'Mean firing rate: {spikes.mean():.3f}')

# Raster plot
n_neurons = 40
n = min(n_neurons, spikes.shape[2])

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')

for neuron in range(n):
    spike_times = np.where(spikes[:, 0, neuron] == 1)[0]
    ax.scatter(spike_times, np.full_like(spike_times, neuron),
               marker='|', s=20, color=plt.cm.plasma(neuron / n), linewidths=0.8)

ax.set_xlabel('Time Step', color='white', fontsize=11)
ax.set_ylabel('Neuron Index', color='white', fontsize=11)
ax.set_title(f'Spike Train Raster — Healthy Sample (first {n} neurons)', color='white', fontsize=13, fontweight='bold')
ax.tick_params(colors='white')
plt.tight_layout()
plt.show()

## Baseline Model - Random Forest

In [ ]:
from sklearn.model_selection import train_test_split
from src.baseline_model import train_baseline, evaluate_model, get_feature_importance

X_tr, X_te, y_tr, y_te = train_test_split(
    X_features, y_feat, test_size=0.25, random_state=42, stratify=y_feat
)

rf_model = train_baseline(X_tr, y_tr, model_type='random_forest', seed=42)
rf_metrics = evaluate_model(rf_model, X_te, y_te, model_name='Random Forest')

print('\nRandom Forest Metrics:')
for k, v in rf_metrics.items():
    print(f'  {k:<12}: {v:.4f}')

In [ ]:
# Feature importance
importances = get_feature_importance(rf_model, feat_names, top_n=15)

fig, ax = plt.subplots(figsize=(10, 5))
names  = list(importances.keys())
values = list(importances.values())
bars   = ax.barh(names[::-1], values[::-1], color='#3498db', alpha=0.85)
ax.set_xlabel('Feature Importance', fontsize=11)
ax.set_title('Top 15 Most Important Features (Random Forest)', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## SNN Training

In [ ]:
import torch
from src.snn_model import EEGDepression_SNN
from src.train import train_snn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_features = X_features.shape[1]

# Split data
n_total = len(y_feat)
indices = np.arange(n_total)
np.random.shuffle(indices)
test_idx  = indices[:int(n_total * 0.25)]
train_idx = indices[int(n_total * 0.25):]

spikes_train = spikes[:, train_idx, :]
spikes_test  = spikes[:, test_idx,  :]
y_snn_train  = y_feat[train_idx]
y_snn_test   = y_feat[test_idx]

snn_model = EEGDepression_SNN(n_inputs=n_features, n_hidden=128, n_outputs=2, beta=0.9)
print(f'SNN parameters: {sum(p.numel() for p in snn_model.parameters()):,}')

history = train_snn(
    snn_model, spikes_train, y_snn_train,
    spikes_val=spikes_test, y_val=y_snn_test,
    n_epochs=25, batch_size=16, lr=1e-3, device=device, seed=42
)

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
epochs = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs, history['train_loss'], '#3498db', lw=2, label='Train')
ax1.plot(epochs, history['val_loss'],   '#e74c3c', lw=2, label='Val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history['train_acc'], '#2ecc71', lw=2, label='Train')
ax2.plot(epochs, history['val_acc'],   '#f39c12', lw=2, label='Val')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylim(0, 1)
ax2.legend(); ax2.grid(True, alpha=0.3)

fig.suptitle('SNN Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Patient Adaptation

In [ ]:
from src.evaluate import evaluate_snn
from src.train import patient_adaptation

# Evaluate before adaptation
_, scores_before, metrics_before = evaluate_snn(
    snn_model, spikes_test, y_snn_test, device=device, model_name='SNN (Before)'
)

# Simulate new patient (last 5 from test set)
spikes_patient = spikes_test[:, :5, :]
y_patient      = y_snn_test[:5]

adapted_model, _ = patient_adaptation(
    snn_model, spikes_patient, y_patient, n_epochs=3, lr=5e-4, device=device
)

# Evaluate after adaptation
_, scores_after, metrics_after = evaluate_snn(
    adapted_model, spikes_test, y_snn_test, device=device, model_name='SNN (After)'
)

# Print comparison table
from src.evaluate import print_adaptation_table
print_adaptation_table(metrics_before, metrics_after)

## Model Comparison

In [ ]:
from src.evaluate import compute_metrics

rf_preds = rf_model.predict(X_te)
all_metrics = {
    'Random Forest': rf_metrics,
    'SNN (Before)':  metrics_before,
    'SNN (Adapted)': metrics_after,
}

# Summary table
summary_df = pd.DataFrame(all_metrics).T
print('\n📊 Model Comparison Summary:')
print(summary_df.to_string())

# Bar chart
metrics_to_show = ['accuracy', 'f1', 'precision', 'recall']
x = np.arange(len(metrics_to_show))
width = 0.25
colors_map = ['#3498db', '#e74c3c', '#2ecc71']

fig, ax = plt.subplots(figsize=(10, 5))
for i, (name, mets) in enumerate(all_metrics.items()):
    vals = [mets.get(m, 0) for m in metrics_to_show]
    bars = ax.bar(x + (i - 1) * width, vals, width, label=name, color=colors_map[i], alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.2f}', ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([m.capitalize() for m in metrics_to_show], fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_title('Model Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()